# Ensemble and Submission Generation

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '..')

from src.models.ensemble import weighted_average_ensemble, optimize_weights
from src.submission.generator import generate_submission, postprocess_predictions
from src.evaluation.metrics import rmsle
from src.config import ENSEMBLE_WEIGHTS, SEED

print('Imports successful.')

In [ ]:
# Demonstrate optimize_weights with synthetic predictions
np.random.seed(SEED)
n_samples = 500

y_true = np.random.rand(n_samples) * 100

# Simulate model predictions with different noise levels
pred_lgbm = np.clip(y_true + np.random.randn(n_samples) * 5, 0, None)
pred_xgb = np.clip(y_true + np.random.randn(n_samples) * 7, 0, None)
pred_cat = np.clip(y_true + np.random.randn(n_samples) * 9, 0, None)

print('=== Optimizing ensemble weights on hold-out set ===')
print()

# Optimize weights
optimal_weights = optimize_weights(
    predictions=[pred_lgbm, pred_xgb, pred_cat],
    y_true=y_true
)

print(f'Default weights:  LGBM={ENSEMBLE_WEIGHTS["lgbm"]}, XGB={ENSEMBLE_WEIGHTS["xgb"]}, CatBoost={ENSEMBLE_WEIGHTS["catboost"]}')
print(f'Optimized weights: LGBM={optimal_weights[0]:.3f}, XGB={optimal_weights[1]:.3f}, CatBoost={optimal_weights[2]:.3f}')
print()

# Compare RMSLE
default_ensemble = weighted_average_ensemble(
    [pred_lgbm, pred_xgb, pred_cat],
    [ENSEMBLE_WEIGHTS['lgbm'], ENSEMBLE_WEIGHTS['xgb'], ENSEMBLE_WEIGHTS['catboost']]
)
optimal_ensemble = weighted_average_ensemble(
    [pred_lgbm, pred_xgb, pred_cat],
    optimal_weights
)

print(f'Default weights RMSLE:   {rmsle(y_true, default_ensemble):.4f}')
print(f'Optimized weights RMSLE: {rmsle(y_true, optimal_ensemble):.4f}')

In [ ]:
# Demonstrate generate_submission with synthetic data
from pathlib import Path

DATA_DIR = Path('../data/raw')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('Sample submission shape:', sample_submission.shape)
print('Columns:', list(sample_submission.columns))
display(sample_submission.head())

# Create synthetic predictions matching submission IDs
n_test = len(sample_submission)
synthetic_preds = np.random.rand(n_test) * 50

# Post-process predictions
processed_preds = postprocess_predictions(synthetic_preds)
print(f'\nSynthetic predictions stats:')
print(f'  Min: {processed_preds.min():.4f}')
print(f'  Max: {processed_preds.max():.4f}')
print(f'  Mean: {processed_preds.mean():.4f}')
print(f'  Zero count: {(processed_preds == 0).sum()}')

# Demonstrate generate_submission (dry run - saves to a temp path)
output_path = Path('../submissions/submission_demo.csv')
output_path.parent.mkdir(exist_ok=True)

submission_df = generate_submission(
    test_ids=sample_submission['id'],
    predictions=processed_preds,
    output_path=output_path
)
print(f'\nSubmission file generated: {output_path}')
display(submission_df.head())

In [ ]:
# Kaggle submission command
print('=== Kaggle Submission Command ===')
print()
kaggle_cmd = (
    'kaggle competitions submit '
    '-c store-sales-time-series-forecasting '\n    '-f submissions/submission.csv '\n    '-m "Ensemble v1"'
)
print(kaggle_cmd)
print()
print('Requirements:')
print('  - Kaggle API credentials configured (~/.kaggle/kaggle.json)')
print('  - submissions/submission.csv file exists')
print('  - Competition rules accepted on Kaggle website')

## Pipeline
1. Train models
2. Optimize weights on hold-out
3. Generate test predictions
4. Post-process and submit